# KAIROS — Démonstration de reproductibilité (Acte 2)

**Rejeu d'un artefact scellé, depuis une source publique — aucun composant propriétaire Kairos.**

> ⚠️ **Ce notebook Colab est un véhicule de reproduction alternatif — ce n'est pas l'environnement de confiance de Kairos.**
> Le replay sur votre propre machine locale reste le chemin de référence, avec la plus grande valeur probatoire. Voir la section "Prérequis" et "Exécution" du dossier complet pour la procédure locale.

Run all cells (Runtime → Run all). ~2-3 minutes. Aucune installation préalable.

Ce notebook télécharge le dataset public BBBP depuis sa source d'origine, reconstruit l'artefact Parquet avec l'enveloppe exacte (Python 3.10, pandas 1.5.3, pyarrow 23.0.0), et compare l'empreinte obtenue à celle scellée le 2026-06-06.

Dossier technique complet : voir VALIDATION_AND_EVIDENCE.html

In [ ]:
#@title Step 1 — Bootstrap micromamba (fournit Python 3.10 quel que soit l'hôte)
!curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj bin/micromamba
!./bin/micromamba create -y -p ./env python=3.10.12 pip -c conda-forge > /dev/null 2>&1
print("Environnement Python 3.10 prêt.")

In [ ]:
#@title Step 2 — Installer l'enveloppe exacte
!./env/bin/pip install -q pandas==1.5.3 pyarrow==23.0.0 "numpy<2"

In [ ]:
#@title Step 3 — Récupérer et vérifier le script de rejeu (fourni par le dossier)
script = r'''#!/usr/bin/env python3
import hashlib
import urllib.request
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

SOURCE = "https://deepchemdata.s3.us-west-1.amazonaws.com/datasets/BBBP.csv"
CSV_SHA256_ATTENDU = "d07a38487aeac5cee5508413e468043ef3097451d2a112701c2d60be9ec6b662"
PARQUET_SHA256_SCELLE = "4621ac8d5d4a728a169fb4d5b8c35682b954928a019d575e3eca140cd563489f"

def sha256(path):
    return hashlib.sha256(open(path, "rb").read()).hexdigest()

def main():
    print("pandas", pd.__version__, "| pyarrow", pa.__version__)
    urllib.request.urlretrieve(SOURCE, "BBBP.csv")
    csv_h = sha256("BBBP.csv")
    print("CSV telecharge :", csv_h)
    print("CSV attendu    :", CSV_SHA256_ATTENDU)
    assert csv_h == CSV_SHA256_ATTENDU, "Le CSV source ne correspond pas."
    df = pd.read_csv("BBBP.csv", low_memory=False)
    print("lignes :", len(df))
    t = pa.Table.from_pandas(df)
    pq.write_table(t, "bbbp.parquet", compression="snappy")
    obtenu = sha256("bbbp.parquet")
    print("parquet obtenu :", obtenu)
    print("parquet scelle :", PARQUET_SHA256_SCELLE)
    print()
    print("=" * 50)
    if obtenu == PARQUET_SHA256_SCELLE:
        print("PASS — YOUR REPLAY")
        print("L'artefact reconstruit correspond exactement a la reference scellee.")
    else:
        print("FAIL — YOUR REPLAY")
        print("DIVERGENCE : l'empreinte obtenue ne correspond pas a la reference.")
        print("Ne pas interpreter silencieusement. Voir section 08 du dossier complet.")
    print("=" * 50)
    print()
    print("Rappel : ce resultat provient d'un vehicule de reproduction (Colab).")
    print("Le replay sur machine locale reste le chemin de reference.")

if __name__ == "__main__":
    main()
'''
with open("kairos_replay_bbbp.py", "w") as f:
    f.write(script)

import hashlib
got = hashlib.sha256(open("kairos_replay_bbbp.py", "rb").read()).hexdigest()
print("SHA-256 du script :", got)

In [ ]:
#@title Step 4 — Exécuter le rejeu
!./env/bin/python kairos_replay_bbbp.py